# 01. Baseline: признаки только из answer

Первый эксперимент: обучаем модели только на статистических признаках, извлеченных из поля `answer` исходного файла `data.csv`. Признаки описания и изображения здесь не используются.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

## Загрузка данных

In [ ]:
DATA_PATH = Path("../data.csv")

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.info()

## Извлечение текстовых признаков

Если в ответе несколько вариантов через `|`, признаки считаем только по первому варианту.

In [ ]:
VOWELS = set("аеёиоуыэюя")
RARE_LETTERS = set("фщъёцэ")
def extract_answer_features(answer):
    if pd.isna(answer):
        answer = ""

    text = str(answer).lower().strip()
    variants = [part.strip() for part in text.split("|") if part.strip()]
    primary_answer = variants[0] if variants else ""
    compact = primary_answer.replace(" ", "")
    words = [word for word in primary_answer.split() if word]

    letters = [char for char in compact if char.isalpha()]
    cyrillic_letters = [char for char in letters if "а" <= char <= "я" or char == "ё"]
    latin_letters = [char for char in letters if "a" <= char <= "z"]

    len_chars = len(compact)
    vowel_count = sum(char in VOWELS for char in compact)
    consonant_count = sum(char.isalpha() and char not in VOWELS for char in compact)
    rare_count = sum(char in RARE_LETTERS for char in compact)
    unique_chars = len(set(compact))

    return pd.Series(
        {
            "len_chars": len_chars,
            "len_words": len(words),
            "avg_word_len": len_chars / len(words) if words else 0.0,
            "vowel_count": vowel_count,
            "consonant_count": consonant_count,
            "rare_count": rare_count,
            "unique_chars": unique_chars,
            "vowel_ratio": vowel_count / len_chars if len_chars else 0.0,
            "rare_ratio": rare_count / len_chars if len_chars else 0.0,
            "unique_ratio": unique_chars / len_chars if len_chars else 0.0,
            "cyrillic_ratio": len(cyrillic_letters) / len(letters) if letters else 0.0,
            "latin_ratio": len(latin_letters) / len(letters) if letters else 0.0,
            "has_dash": int("-" in text),
            "has_digits": int(any(char.isdigit() for char in primary_answer)),
        }
    )


text_features = df["answer"].apply(extract_answer_features)
text_features.head()

In [ ]:
dataset = pd.concat([df[["answer", "difficulty"]], text_features], axis=1)
dataset.head()

## Обучение моделей

In [ ]:
X = text_features
y = df["difficulty"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

X_train.shape, X_test.shape

In [ ]:
RANDOM_STATE = 42


def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def scaled_model(model):
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", model),
        ]
    )


models = {
    "dummy_mean": DummyRegressor(strategy="mean"),
    "linear_regression": scaled_model(LinearRegression()),
    "ridge": scaled_model(Ridge(alpha=1.0)),
    "lasso": scaled_model(Lasso(alpha=0.001, max_iter=10_000, random_state=RANDOM_STATE)),
    "elastic_net": scaled_model(
        ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10_000, random_state=RANDOM_STATE)
    ),
    "svr_rbf": scaled_model(SVR(kernel="rbf", C=1.0, epsilon=0.05)),
    "knn_5": scaled_model(KNeighborsRegressor(n_neighbors=5, weights="distance")),
    "random_forest": RandomForestRegressor(
        n_estimators=500,
        max_depth=5,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
    ),
    "extra_trees": ExtraTreesRegressor(
        n_estimators=500,
        max_depth=5,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
    ),
    "gradient_boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=2,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
    ),
}


def regression_metrics(y_true, y_pred):
    return {
        "r2": r2_score(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
    }


def evaluate_holdout(models, X_train, X_test, y_train, y_test):
    rows = []
    fitted_models = {}

    for name, model in models.items():
        fitted_model = clone(model)
        fitted_model.fit(X_train, y_train)
        y_pred = fitted_model.predict(X_test)

        rows.append({"model": name, **regression_metrics(y_test, y_pred)})
        fitted_models[name] = fitted_model

    metrics = pd.DataFrame(rows).sort_values("mae").reset_index(drop=True)
    return metrics, fitted_models


holdout_metrics, fitted_models = evaluate_holdout(models, X_train, X_test, y_train, y_test)
holdout_metrics

## Кросс-валидация

На маленьком датасете один train/test split может сильно шуметь, поэтому дополнительно смотрим 5-fold CV.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "rmse": make_scorer(rmse_score, greater_is_better=False),
}

cv_rows = []

for name, model in models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring)
    cv_rows.append(
        {
            "model": name,
            "r2_mean": scores["test_r2"].mean(),
            "r2_std": scores["test_r2"].std(),
            "mae_mean": -scores["test_mae"].mean(),
            "mae_std": scores["test_mae"].std(),
            "rmse_mean": -scores["test_rmse"].mean(),
            "rmse_std": scores["test_rmse"].std(),
        }
    )

cv_metrics = pd.DataFrame(cv_rows).sort_values("mae_mean").reset_index(drop=True)
cv_metrics

## Сравнение holdout и cross-validation

In [ ]:
summary = holdout_metrics.merge(
    cv_metrics,
    on="model",
    how="left",
)

summary[
    [
        "model",
        "mae",
        "r2",
        "mae_mean",
        "mae_std",
        "rmse_mean",
        "r2_mean",
        "r2_std",
    ]
].sort_values("mae_mean")

## Лучшая модель и ошибки на holdout

In [ ]:
best_model_name = cv_metrics.iloc[0]["model"]
best_model = fitted_models[best_model_name]
y_pred = best_model.predict(X_test)

print(f"Best model by CV MAE: {best_model_name}")

predictions = pd.DataFrame(
    {
        "answer": df.loc[y_test.index, "answer"],
        "actual_difficulty": y_test,
        "predicted_difficulty": y_pred,
        "absolute_error": np.abs(y_test - y_pred),
    }
).sort_values("absolute_error", ascending=False)

predictions.head(20)

## Важность признаков лучшей модели

Permutation importance показывает, насколько ухудшается MAE при случайном перемешивании каждого признака.

In [ ]:
importance = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=30,
    random_state=RANDOM_STATE,
    scoring="neg_mean_absolute_error",
)

feature_importance = pd.DataFrame(
    {
        "feature": X.columns,
        "importance_mean": importance.importances_mean,
        "importance_std": importance.importances_std,
    }
).sort_values("importance_mean", ascending=False)

feature_importance

## Распределение реальных и предсказанных значений

Сравнение плотности распределения целевой переменной на тестовой выборке.

In [ ]:
plot_df = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
}).sort_values("Actual").reset_index(drop=True)

plt.figure(figsize=(12, 6))
plt.plot(plot_df["Actual"].values, label="Actual", linewidth=2)
plt.plot(plot_df["Predicted"].values, label="Predicted", alpha=0.7)
plt.title(f"Actual vs Predicted Difficulty (Sorted by Actual, Model: {best_model_name})")
plt.xlabel("Sample Index (sorted by actual difficulty)")
plt.ylabel("Difficulty")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()